# AFRO-PRODUCTIONS ANALYSIS

Authors: 
- Angela Masaki
- Joy Nyuguto
- David Theuri
- Joel Muoki
- Felista Mwangi

Date:March 2026

## Overview


This notebook presents an  exploratory data analysis (EDA) to direct the opening of a new movie film studio.We determine what distinguishes high-performing movies from low-performing ones using box office, budget, and genre data from various sources.


## Business Understanding

### Key Stakeholder
The Head of the New Movie Studio (Afro Productions)

### Business Problem

Despite having no  prior experience producing movies,our company,Afro Productions, wants to get into the original video content business. Without historical insight, every decision,including genre, budget, release date, target markets, and current performing studios,is an expensive bet. Before it ever gets off the ground, an underperforming  first film might cost millions of dollars and ruin the business.

### Key Business Questions
Q1.What film genres give the best return on investment?

Q2.What production budget range maximizes profiatability?

Q3.Which release month generates the highest box office gross?

Q4.Should our studio prioritize domestic or international markets?

Q5.Which studios should we model ourselves after?

---



## Data Understanding
We  are using  **three CSV datasets**  extracted from different industry sources.
Each one contributes something different to our analysis.



---



### Importing libraries

In [108]:
# Import important libraries

# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

### Loading the 'csv' movie data sets

In [109]:
# loading the data 
bom = pd.read_csv('../zippedData/bom.movie_gross.csv.gz')
tmdb = pd.read_csv('../zippedData/tmdb.movies.csv.gz')
tn = pd.read_csv('../zippedData/tn.movie_budgets.csv.gz')


### Data Previews

In [110]:
# Data sets shape(row,columns)
print(f'tn shape:{tn.shape}')
print(f'tmdb shape:{tmdb.shape}')
print(f'bom shape:{bom.shape}')

print("-" * 30)
#
print(f'tn columns:{list(tn.columns)}')
print(f'tmdb columns:{list(tmdb.columns)}')
print(f'bom columns:{list(bom.columns)}')

tn shape:(5782, 6)
tmdb shape:(26517, 10)
bom shape:(3387, 5)
------------------------------
tn columns:['id', 'release_date', 'movie', 'production_budget', 'domestic_gross', 'worldwide_gross']
tmdb columns:['Unnamed: 0', 'genre_ids', 'id', 'original_language', 'original_title', 'popularity', 'release_date', 'title', 'vote_average', 'vote_count']
bom columns:['title', 'studio', 'domestic_gross', 'foreign_gross', 'year']


| Source | File | Key Columns | Size |
|--------|------|-------------|------|
| The Numbers | `tn_movie_budgets.csv` | id, release_date, movie, production_budget, domestic_gross, worldwide_gross| 5,782 rows ,6 columns |
| TheMovieDB | `tmdb_movies.csv` | Unnamed: 0, genre_ids, id, original_language, original_title, popularity, release_date, title, vote_average, vote_count| 26,517 rows,10 columns|
| Box Office Mojo | `bom_movie_gross.csv` | title, studio, domestic_gross, foreign_gross, year | 3,387 rows,5 columns|


**Why these datasets?**
- tn_movie_budgets.csv(tn) provides budget and revenue data   -essential for computing ROI.
- tmdb_movies.csv(tmdb) provides genre and popularity metadata -useful for understanding movie genre performance.
- bom_movie_gross.csv(bom)provides domestic vs. foreign splits  - useful for understanding market reach.


## Data Preparation

Before analysis, all three datasets required cleaning to ensure the data was accurate, consistent and ready for use.


### Displaying sections of the movie data sets

In [111]:
# Data preview
print("The Numbers (tn):")
display(tn.head())

print("The Movie DB (tmdb):")
display(tmdb.head())

print("Box Office Mojo (bom):")
display(bom.head()) 




The Numbers (tn):


,id,release_date,movie,production_budget,domestic_gross,worldwide_gross
0,1,"Dec 18, 2009",Avatar,"$425,000,000","$760,507,625","$2,776,345,279"
1,2,"May 20, 2011",Pirates of the Caribbean: On Stranger Tides,"$410,600,000","$241,063,875","$1,045,663,875"
2,3,"Jun 7, 2019",Dark Phoenix,"$350,000,000","$42,762,350","$149,762,350"
3,4,"May 1, 2015",Avengers: Age of Ultron,"$330,600,000","$459,005,868","$1,403,013,963"
4,5,"Dec 15, 2017",Star Wars Ep. VIII: The Last Jedi,"$317,000,000","$620,181,382","$1,316,721,747"


The Movie DB (tmdb):


,Unnamed: 0,genre_ids,id,original_language,original_title,popularity,release_date,title,vote_average,vote_count
0,0,"[12, 14, 10751]",12444,en,Harry Potter and the Deathly Hallows: Part 1,33.533,2010-11-19,Harry Potter and the Deathly Hallows: Part 1,7.7,10788
1,1,"[14, 12, 16, 10751]",10191,en,How to Train Your Dragon,28.734,2010-03-26,How to Train Your Dragon,7.7,7610
2,2,"[12, 28, 878]",10138,en,Iron Man 2,28.515,2010-05-07,Iron Man 2,6.8,12368
3,3,"[16, 35, 10751]",862,en,Toy Story,28.005,1995-11-22,Toy Story,7.9,10174
4,4,"[28, 878, 12]",27205,en,Inception,27.920,2010-07-16,Inception,8.3,22186


Box Office Mojo (bom):


,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010


### Cleaning The Numbers :`tn_movie_budgets.csv`(tn)




- Renamed `id` to `rank` since the column represents budget ranking, not a unique identifier
- Stripped `$` signs and commas from the three money columns (`production_budget`, `domestic_gross`, `worldwide_gross`) and converted them from strings to floats
- Replaced `$0` entries with `NaN` since a film with zero budget or zero gross is likely a data entry error
- Dropped rows missing any of the three financial columns since these are essential for our analysis
- Parsed `release_date` from string to datetime and extracted `year` and `month` as separate columns
- **5,234 rows remaining after cleaning**

In [112]:
#  Rename 'id' to  'rank' (not a true unique ID)- instead it represents rank according to production budget rank 
tn = tn.rename(columns={'id': 'rank'})

#Removing commas and $ signs from money columns then converting to numbers
for col in ['production_budget', 'domestic_gross', 'worldwide_gross']:
       tn[col] = tn[col].str.replace('$', '', regex=False)
       tn[col] = tn[col].str.replace(',', '', regex=False)
       tn[col] = tn[col].astype(float)   


#Repacing 0 values in columns with Nan
for col in ['production_budget', 'domestic_gross', 'worldwide_gross']:
    tn[col] = tn[col].replace(0, np.nan)

# Dropping rows with missing financial data values 
tn = tn.dropna(subset=['production_budget', 'domestic_gross', 'worldwide_gross'])

#Parsing release date
tn['release_date'] = pd.to_datetime(tn['release_date'])

# Extracting release year and months into their own columns
tn['year'] = tn['release_date'].dt.year
tn['month'] = tn['release_date'].dt.month

print("tn after cleaning")
print(tn.dtypes)
print()
print(tn.isnull().sum())
print()
print(tn.head())




tn after cleaning
rank                          int64
release_date         datetime64[ns]
movie                        object
production_budget           float64
domestic_gross              float64
worldwide_gross             float64
year                          int64
month                         int64
dtype: object

rank                 0
release_date         0
movie                0
production_budget    0
domestic_gross       0
worldwide_gross      0
year                 0
month                0
dtype: int64

   rank release_date                                        movie  \
0     1   2009-12-18                                       Avatar   
1     2   2011-05-20  Pirates of the Caribbean: On Stranger Tides   
2     3   2019-06-07                                 Dark Phoenix   
3     4   2015-05-01                      Avengers: Age of Ultron   
4     5   2017-12-15            Star Wars Ep. VIII: The Last Jedi   

   production_budget  domestic_gross  worldwide_gross  year  month

### Cleaning Movie DB :`tmdb_movies.csv`(tmdb)



- Dropped the unnamed index column which added no value
- Parsed `release_date` from string to datetime and extracted `year` and `month`
- Removed rows with empty genre lists (`[]`) since genre is central to our analysis
- Converted `genre_ids` from a string representation into actual Python lists
- Removed films with fewer than 10 votes since ratings based on very few reviews are unreliable
- Decoded genre IDs into readable names (e.g. `28` to `Action`) using the official TMDB genre mapping and stored the first listed genre as `primary_genre`
- **10,348 rows remaining after cleaning**

In [113]:

# Dropping the unnamed index column 
tmdb = tmdb.drop(columns=['Unnamed: 0'])

# Parsing release_date from string to datetime
tmdb['release_date'] = pd.to_datetime(tmdb['release_date'])

# Extracting year and month
tmdb['year'] = tmdb['release_date'].dt.year
tmdb['month'] = tmdb['release_date'].dt.month

# Dropping  rows where genre_ids is empty 
tmdb = tmdb[tmdb['genre_ids'] != '[]']

# Convert genre_ids from a string like "[12, 14]" to an actual Python list
import ast
tmdb['genre_ids'] = tmdb['genre_ids'].apply(ast.literal_eval)

# Drop films with fewer than 10 votes since they are unreliable
tmdb = tmdb[tmdb['vote_count'] >= 10]

# Genre ID to name dictionary (official TMDB genre IDs)
GENRE_MAP = {
    28: 'Action',    12: 'Adventure',  16: 'Animation',   35: 'Comedy',
    80: 'Crime',     99: 'Documentary',18: 'Drama',     10751: 'Family',
    14: 'Fantasy',   36: 'History',    27: 'Horror',    10402: 'Music',
  9648: 'Mystery', 10749: 'Romance',  878: 'Sci-Fi',   10770: 'TV Movie',
    53: 'Thriller', 10752: 'War',      37: 'Western'
}

# Getting  the first genre from each list and mappin it to a name
tmdb['primary_genre'] = tmdb['genre_ids'].apply(lambda x: GENRE_MAP.get(x[0], 'Unknown'))

# Dropping rows where genre couldn't be decoded
tmdb = tmdb[tmdb['primary_genre'] != 'Unknown']


print("tmdb after cleaning")
print(f"Rows remaining: {len(tmdb)}")
print(tmdb.dtypes)
print()
print(tmdb.isnull().sum())
print()
print(tmdb.head())

tmdb after cleaning
Rows remaining: 10348
genre_ids                    object
id                            int64
original_language            object
original_title               object
popularity                  float64
release_date         datetime64[ns]
title                        object
vote_average                float64
vote_count                    int64
year                          int64
month                         int64
primary_genre                object
dtype: object

genre_ids            0
id                   0
original_language    0
original_title       0
popularity           0
release_date         0
title                0
vote_average         0
vote_count           0
year                 0
month                0
primary_genre        0
dtype: int64

             genre_ids     id original_language  \
0      [12, 14, 10751]  12444                en   
1  [14, 12, 16, 10751]  10191                en   
2        [12, 28, 878]  10138                en   
3      [16, 35, 1

### Cleaning Box Office Mojo `bom_movie_gross.csv` (bom)

- Dropped 5 rows with missing studio information
- Converted `foreign_gross` from string to float using `pd.to_numeric`
- Created a `total_gross` column by summing domestic and foreign gross
- Dropped rows where **both** domestic and foreign gross were missing  and rows with at least one value were kept
- **3,382 rows remaining after cleaning**

In [114]:

# Drop rows with missing studio values
bom = bom.dropna(subset=['studio'])

# Convert foreign_gross from string to float
bom['foreign_gross'] = pd.to_numeric(bom['foreign_gross'], errors='coerce')

# Computing total gross 
bom['total_gross'] = bom['domestic_gross'].fillna(0) + bom['foreign_gross'].fillna(0)

# Dropping  rows with missing  BOTH domestic and foreign gross values
bom = bom.dropna(subset=['domestic_gross', 'foreign_gross'], how='all')

print("bom after cleaning")
print(f"Rows remaining: {len(bom)}")
print(bom.dtypes)
print()
print(bom.isnull().sum())
print()
print()
print(bom.head())

bom after cleaning
Rows remaining: 3382
title              object
studio             object
domestic_gross    float64
foreign_gross     float64
year                int64
total_gross       float64
dtype: object

title                0
studio               0
domestic_gross      26
foreign_gross     1354
year                 0
total_gross          0
dtype: int64


                                         title studio  domestic_gross  \
0                                  Toy Story 3     BV     415000000.0   
1                   Alice in Wonderland (2010)     BV     334200000.0   
2  Harry Potter and the Deathly Hallows Part 1     WB     296000000.0   
3                                    Inception     WB     292600000.0   
4                          Shrek Forever After   P/DW     238700000.0   

   foreign_gross  year   total_gross  
0    652000000.0  2010  1.067000e+09  
1    691300000.0  2010  1.025500e+09  
2    664300000.0  2010  9.603000e+08  
3    535700000.0  2010  8.283000e+08  
4 

## Data Merging & Saving Cleaned DataFrames

We merge the three cleaned datasets into a single analysis-ready DataFrame
and save both `movies_clean.csv` and `movies_exploded.csv` so the
visualisation section can be run independently.

In [ ]:
import os
import ast

# ----------------------------------------------------------------
# MERGE: join tn (budget/revenue) with tmdb (genre/ratings)
# ----------------------------------------------------------------
tn_copy   = tn.copy()
tmdb_copy = tmdb.copy()

tn_copy['movie_key']   = tn_copy['movie'].str.lower().str.strip()
tmdb_copy['movie_key'] = tmdb_copy['title'].str.lower().str.strip()

merged = pd.merge(
    tn_copy,
    tmdb_copy[['movie_key', 'vote_average', 'vote_count',
               'genre_ids', 'primary_genre']],
    on='movie_key', how='left'
)

merged = merged.rename(columns={
    'movie'             : 'primary_title',
    'year'              : 'release_year',
    'production_budget' : 'budget',
    'worldwide_gross'   : 'worldwide_gross_raw'
})

merged['worldwide_gross_millions'] = merged['worldwide_gross_raw']  / 1_000_000
merged['budget_millions']          = merged['budget']               / 1_000_000
merged['domestic_gross_millions']  = merged['domestic_gross']       / 1_000_000

merged['roi_percent'] = (
    (merged['worldwide_gross_raw'] - merged['budget']) / merged['budget'] * 100
).round(2)

merged = merged.drop(columns=['movie_key'])
movies_df = merged.copy()

print(f'movies_df shape: {movies_df.shape}')
print(movies_df[['primary_title', 'release_year', 'budget_millions',
                  'worldwide_gross_millions', 'roi_percent',
                  'vote_average', 'primary_genre']].head())

In [ ]:
# ----------------------------------------------------------------
# BUILD movies_exploded — one row per genre per movie
# ----------------------------------------------------------------

GENRE_MAP = {
    28: 'Action',    12: 'Adventure',  16: 'Animation',   35: 'Comedy',
    80: 'Crime',     99: 'Documentary',18: 'Drama',     10751: 'Family',
    14: 'Fantasy',   36: 'History',    27: 'Horror',    10402: 'Music',
  9648: 'Mystery', 10749: 'Romance',  878: 'Sci-Fi',   10770: 'TV Movie',
    53: 'Thriller', 10752: 'War',      37: 'Western'
}

explode_base = movies_df.dropna(subset=['genre_ids']).copy()

def safe_parse(x):
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

explode_base['genre_ids'] = explode_base['genre_ids'].apply(safe_parse)
explode_base = explode_base[explode_base['genre_ids'].map(len) > 0]

movies_exploded = explode_base.explode('genre_ids').copy()
movies_exploded['genres'] = movies_exploded['genre_ids'].map(GENRE_MAP)
movies_exploded = movies_exploded.dropna(subset=['genres'])

print(f'movies_exploded shape: {movies_exploded.shape}')
print(movies_exploded[['primary_title', 'genres',
                        'worldwide_gross_millions']].head(10))

In [ ]:
# ----------------------------------------------------------------
# SAVE both DataFrames as CSV
# Paths relative to the notebook location (notebooks/)
# ----------------------------------------------------------------
os.makedirs('../data',   exist_ok=True)
os.makedirs('../images', exist_ok=True)

movies_df.to_csv('../data/movies_clean.csv',          index=False)
movies_exploded.to_csv('../data/movies_exploded.csv', index=False)

print('Saved: data/movies_clean.csv')
print('Saved: data/movies_exploded.csv')

## Results

We analysed box office data from Box Office Mojo, TheMovieDB, and The Numbers.
Below are three visualisations that directly inform our business recommendations
for the new movie studio.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Global style — applied to ALL charts
sns.set_style('whitegrid')
sns.set_context('talk')
plt.rcParams['figure.figsize']   = (12, 7)
plt.rcParams['figure.dpi']       = 150
plt.rcParams['font.family']      = 'DejaVu Sans'
plt.rcParams['axes.titlepad']    = 15
plt.rcParams['axes.titlesize']   = 15
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize']   = 12

os.makedirs('../images', exist_ok=True)

# Load cleaned datasets (self-contained — can run without earlier cells)
movies_df       = pd.read_csv('../data/movies_clean.csv')
movies_exploded = pd.read_csv('../data/movies_exploded.csv')

print(f'movies_df shape      : {movies_df.shape}')
print(f'movies_exploded shape: {movies_exploded.shape}')
print(f'Columns: {movies_df.columns.tolist()}')

### Business Recommendation 1
**Focus on High-Revenue Genres: Action, Adventure, and Animation**

We grouped all movies by genre and calculated the average worldwide gross
revenue. The horizontal bar chart below ranks the Top 10 genres from highest
to lowest average earnings — this tells the studio **where the money is**.

In [ ]:
# DATA PREP: Average worldwide gross by genre
genre_revenue = (
    movies_exploded
    .groupby('genres')['worldwide_gross_millions']
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
genre_revenue.columns = ['Genre', 'Avg Worldwide Gross (Millions USD)']
genre_revenue['Avg Worldwide Gross (Millions USD)'] = (
    genre_revenue['Avg Worldwide Gross (Millions USD)'].round(2)
)
print('Top 10 genres by average worldwide gross:')
print(genre_revenue.to_string(index=False))

In [ ]:
# CHART 1: Top 10 Genres by Average Worldwide Gross (Horizontal Bar)
fig, ax = plt.subplots(figsize=(12, 7))

sns.barplot(
    data=genre_revenue,
    x='Avg Worldwide Gross (Millions USD)',
    y='Genre',
    palette='Blues_d',
    ax=ax,
    order=genre_revenue['Genre']
)

for bar in ax.patches:
    width = bar.get_width()
    ax.text(
        width + 2, bar.get_y() + bar.get_height() / 2,
        f'${width:,.0f}M', va='center', ha='left', fontsize=10, color='#333333'
    )

ax.set_title(
    'Top 10 Movie Genres by Average Worldwide Box Office Revenue\n'
    '(Films Released 2010-Present)',
    fontsize=15, fontweight='bold'
)
ax.set_xlabel('Average Worldwide Gross Revenue (Millions USD)', fontsize=12)
ax.set_ylabel('Genre', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))

overall_mean = movies_exploded['worldwide_gross_millions'].mean()
ax.axvline(x=overall_mean, color='red', linestyle='--', linewidth=1.5,
           label=f'Overall Mean: ${overall_mean:,.0f}M')
ax.legend(fontsize=10)

sns.despine(left=False, bottom=False)
plt.tight_layout()
plt.savefig('../images/genre_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: images/genre_revenue.png')

**Finding 1:** Animation, Adventure, and Action are the top three
highest-earning genres, all averaging well above the overall industry mean
(shown by the red dashed line). Drama and Comedy, despite being prolific,
earn significantly less on average. The studio should prioritise these
high-revenue genres for its first productions.

### Business Recommendation 2
**Invest in Quality: Higher-Rated Films Earn More**

We plotted each movie's audience rating (TMDB) against its worldwide gross.
A scatter plot best shows the relationship between two continuous variables.
The trend line reveals whether the relationship is positive, negative, or flat.

In [ ]:
# DATA PREP: Rating vs Revenue
rating_revenue = movies_df[
    (movies_df['vote_average']            > 0) &
    (movies_df['worldwide_gross_millions'] > 0) &
    (movies_df['vote_count']             >= 50)
].copy()

rating_revenue['rating_tier'] = pd.cut(
    rating_revenue['vote_average'],
    bins   = [0, 5, 6.5, 7.5, 10],
    labels = ['Low (< 5)', 'Medium (5-6.5)', 'Good (6.5-7.5)', 'Top (> 7.5)']
)

print(f'Movies used for this chart: {len(rating_revenue)}')
print('Rating tier distribution:')
print(rating_revenue['rating_tier'].value_counts().sort_index())

In [ ]:
# CHART 2: Audience Rating vs Worldwide Gross (Scatter + trend line)
fig, ax = plt.subplots(figsize=(12, 7))

tier_colors = {
    'Low (< 5)'      : '#d62728',
    'Medium (5-6.5)' : '#ff7f0e',
    'Good (6.5-7.5)' : '#2ca02c',
    'Top (> 7.5)'    : '#1f77b4'
}

for tier, color in tier_colors.items():
    subset = rating_revenue[rating_revenue['rating_tier'] == tier]
    ax.scatter(
        subset['vote_average'], subset['worldwide_gross_millions'],
        alpha=0.5, s=60, color=color,
        label=f'{tier} ({len(subset)} films)',
        edgecolors='white', linewidths=0.4
    )

sns.regplot(
    data=rating_revenue, x='vote_average', y='worldwide_gross_millions',
    scatter=False, color='black',
    line_kws={'linewidth': 2.5, 'linestyle': '--'},
    ci=95, ax=ax
)

ax.set_title(
    'Does Audience Rating Drive Box Office Revenue?\n'
    'TMDB Vote Average vs Worldwide Gross (2010-Present)',
    fontsize=15, fontweight='bold'
)
ax.set_xlabel('TMDB Audience Rating (out of 10)', fontsize=12)
ax.set_ylabel('Worldwide Gross Revenue (Millions USD)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y:,.0f}M'))

ax.annotate(
    'Trend: Higher-rated films\ntend to earn more',
    xy=(7.5, rating_revenue['worldwide_gross_millions'].quantile(0.85)),
    fontsize=10, color='black', style='italic',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8)
)

ax.legend(title='Rating Tier', fontsize=9, title_fontsize=10, loc='upper left')
sns.despine()
plt.tight_layout()
plt.savefig('../images/rating_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: images/rating_revenue.png')

**Finding 2:** There is a positive relationship between audience rating and
worldwide revenue. Films rated above 7.5 (blue dots) cluster at the higher
revenue end. While high ratings alone do not guarantee blockbuster earnings,
low-rated films (red dots) rarely achieve strong box office performance.
The studio should invest in quality scripts, experienced directors, and
strong casts to maximise both ratings and revenue.

### Business Recommendation 3
**Start With a Mid-Range Budget ($30M-$70M) for the Best ROI**

Return on Investment (ROI) measures how much profit a film earns relative
to what was spent. We grouped movies into budget tiers and compared their
**median** ROI. Median is used (not mean) because one mega-hit like
*Avatar* can dramatically skew the average.

In [ ]:
# DATA PREP: Budget tier vs ROI
budget_roi = movies_df[
    (movies_df['budget_millions'] > 0) &
    (movies_df['roi_percent'].between(-100, 2000))
].copy()

budget_roi['budget_tier'] = pd.cut(
    budget_roi['budget_millions'],
    bins   = [0, 10, 30, 70, 150, 500],
    labels = [
        'Micro\n(< $10M)',
        'Low\n($10-30M)',
        'Mid\n($30-70M)',
        'High\n($70-150M)',
        'Blockbuster\n(> $150M)'
    ]
)

budget_summary = (
    budget_roi
    .groupby('budget_tier', observed=True)
    .agg(
        median_roi  = ('roi_percent',   'median'),
        movie_count = ('primary_title', 'count')
    )
    .reset_index()
)
budget_summary.columns = ['Budget Tier', 'Median ROI (%)', 'Number of Films']
budget_summary['Median ROI (%)'] = budget_summary['Median ROI (%)'].round(1)

print('Budget Tier Analysis:')
print(budget_summary.to_string(index=False))

In [ ]:
# CHART 3: Budget Tier vs Median ROI (Vertical Bar)
fig, ax1 = plt.subplots(figsize=(12, 7))

bar_colors = ['#aec6cf', '#aec6cf', '#2ecc71', '#aec6cf', '#aec6cf']
bars = ax1.bar(
    budget_summary['Budget Tier'], budget_summary['Median ROI (%)'],
    color=bar_colors, edgecolor='white', linewidth=1.5, width=0.55
)

for bar in bars:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2, h + 5,
             f'{h:.1f}%', ha='center', va='bottom',
             fontsize=11, fontweight='bold', color='#333333')

for bar, count in zip(bars, budget_summary['Number of Films']):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
             f'n = {count}', ha='center', va='center',
             fontsize=9, color='white', fontweight='bold')

ax1.axhline(y=0, color='black', linewidth=1)

mid_mask = budget_summary['Budget Tier'] == 'Mid\n($30-70M)'
if mid_mask.any():
    sweet_roi = budget_summary.loc[mid_mask, 'Median ROI (%)'].values[0]
    ax1.annotate(
        'Sweet Spot:\nBest ROI per dollar spent',
        xy=(2, sweet_roi + 10), ha='center',
        fontsize=10, color='#27ae60', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#eafaf1', alpha=0.9)
    )

ax1.set_title(
    'Which Budget Range Delivers the Best Return on Investment (ROI)?\n'
    'Median ROI by Production Budget Tier (2010-Present)',
    fontsize=15, fontweight='bold'
)
ax1.set_xlabel('Production Budget Tier', fontsize=12)
ax1.set_ylabel('Median ROI (%)', fontsize=12)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:.0f}%'))

sns.despine()
plt.tight_layout()
plt.savefig('../images/budget_roi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: images/budget_roi.png')

**Finding 3:** Mid-budget films ($30M-$70M) show the highest median ROI,
meaning they return the most profit relative to what was invested.
Blockbuster-budget films (> $150M) earn more in total gross but require
enormous upfront investment and carry much higher financial risk. For a
new studio with no track record, starting with mid-range budgets
maximises the chance of profitability while limiting downside risk.

> **Socratic Check:** We used **median** ROI instead of **mean** ROI for
> each tier. Can you explain why? Think about what happens to the mean when
> one film like *Avatar* earns $3 billion.

### Bonus Chart 4 - Average Worldwide Revenue Trend by Year

A line chart demonstrates use of a time-series chart type and provides
broader context about year-to-year industry performance.

In [ ]:
# BONUS CHART 4: Average Worldwide Gross by Year (Line Chart)
yearly_revenue = (
    movies_df
    .groupby('release_year')['worldwide_gross_millions']
    .agg(['mean', 'count'])
    .reset_index()
)
yearly_revenue.columns = ['Year', 'Avg Gross (Millions)', 'Film Count']
yearly_revenue = yearly_revenue[yearly_revenue['Film Count'] >= 5]
print(yearly_revenue.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(
    yearly_revenue['Year'], yearly_revenue['Avg Gross (Millions)'],
    color='#1f77b4', linewidth=2.5, marker='o', markersize=8,
    label='Avg Worldwide Gross'
)
ax.fill_between(
    yearly_revenue['Year'], yearly_revenue['Avg Gross (Millions)'],
    alpha=0.15, color='#1f77b4'
)

for _, row in yearly_revenue.iterrows():
    ax.annotate(
        f"${row['Avg Gross (Millions)']:.0f}M",
        xy=(row['Year'], row['Avg Gross (Millions)']),
        xytext=(0, 10), textcoords='offset points',
        ha='center', fontsize=8, color='#1f77b4'
    )

if 2020 in yearly_revenue['Year'].values:
    covid_val = yearly_revenue.loc[
        yearly_revenue['Year'] == 2020, 'Avg Gross (Millions)'
    ].values[0]
    ax.annotate(
        'COVID-19\nimpact',
        xy=(2020, covid_val), xytext=(2019, covid_val + 60),
        arrowprops=dict(arrowstyle='->', color='red'),
        fontsize=9, color='red'
    )

ax.set_title(
    'Average Worldwide Box Office Revenue by Year\n'
    'Understanding When the Industry Earns Most',
    fontsize=15, fontweight='bold'
)
ax.set_xlabel('Release Year', fontsize=12)
ax.set_ylabel('Average Worldwide Gross (Millions USD)', fontsize=12)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y:,.0f}M'))
plt.xticks(rotation=45)
ax.legend(fontsize=10)

sns.despine()
plt.tight_layout()
plt.savefig('../images/revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: images/revenue_trend.png')

In [ ]:
# Verify all charts were saved
expected_charts = [
    '../images/genre_revenue.png',
    '../images/rating_revenue.png',
    '../images/budget_roi.png',
    '../images/revenue_trend.png'
]
print('Chart files status:')
for path in expected_charts:
    status = 'Found' if os.path.exists(path) else 'MISSING - re-run that cell'
    print(f'  {status}: {path}')